In [1]:
import pandas as pd
import numpy as np
import os
import time
import glob
import pyarrow as pa
import pyarrow.parquet as pq

# 1. Dọn dẹp các file cũ
files_to_delete = glob.glob("../hdfs/data-input/drug_part_*") \
+ glob.glob("../hdfs/data-input/*.parquet") \
+ glob.glob("../hdfs/data-input/*.csv") 

for f in files_to_delete:
    try:
        os.remove(f)
        print(f"Đã xóa: {f}")
    except:
        pass

def generate_full_dataset_parquet(total_rows, chunk_size, output_file):
    start_time = time.time()
    
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    num_chunks = total_rows // chunk_size
    
    writer = None

    for i in range(num_chunks):
        # Tính toán start_id cho mỗi chunk
        # Ví dụ: chunk 0 bắt đầu từ 1, chunk 1 bắt đầu từ 5,000,001
        start_id = i * chunk_size + 1
        row_ids = np.arange(start_id, start_id + chunk_size, dtype=np.int64)

        age = np.random.randint(15, 75, size=chunk_size)
        na_to_k = np.random.uniform(6.0, 35.0, size=chunk_size)
        bp = np.random.choice(['HIGH', 'LOW', 'NORMAL'], size=chunk_size)
        cholesterol = np.random.choice(['HIGH', 'NORMAL'], size=chunk_size)
        sex = np.random.choice(['F', 'M'], size=chunk_size)
        
        drug = np.where(na_to_k > 15, 'DrugY', 
               np.where((age > 50) & (bp == 'HIGH'), 'DrugB',
               np.where(bp == 'HIGH', 'DrugA',
               np.where(bp == 'LOW', 'DrugC', 'DrugX'))))

        # Thêm cột row_id vào DataFrame
        df_chunk = pd.DataFrame({
            'row_id': row_ids,
            'Age': age.astype(np.int32), 
            'Sex': sex, 
            'BP': bp,
            'Cholesterol': cholesterol, 
            'Na_to_K': na_to_k.round(3).astype(np.float32),
            'Drug': drug
        })

        # Chuyển đổi DataFrame sang PyArrow Table
        table = pa.Table.from_pandas(df_chunk)

        # Khởi tạo ParquetWriter
        if writer is None:
            writer = pq.ParquetWriter(output_file, table.schema, compression='snappy')
        
        # Ghi chunk vào file
        writer.write_table(table)
        print(f"Đã ghi xong: {(i + 1) * chunk_size} dòng (row_id đến {row_ids[-1]})")

    if writer:
        writer.close()

    end_time = time.time()
    print(f"File lưu tại: {output_file}")
    print(f"Tổng thời gian: {(end_time - start_time)/60:.2f} phút")

# Cấu hình tham số
TOTAL_ROWS = 50_000_000 
CHUNK_SIZE = 5_000_000 
OUTPUT_FILE = "../hdfs/data-input/drug200.parquet"

generate_full_dataset_parquet(TOTAL_ROWS, CHUNK_SIZE, OUTPUT_FILE)

Đã xóa: ../hdfs/data-input/drug200.parquet
Đã ghi xong: 5000000 dòng (row_id đến 5000000)
Đã ghi xong: 10000000 dòng (row_id đến 10000000)
Đã ghi xong: 15000000 dòng (row_id đến 15000000)
Đã ghi xong: 20000000 dòng (row_id đến 20000000)
Đã ghi xong: 25000000 dòng (row_id đến 25000000)
Đã ghi xong: 30000000 dòng (row_id đến 30000000)
Đã ghi xong: 35000000 dòng (row_id đến 35000000)
Đã ghi xong: 40000000 dòng (row_id đến 40000000)
Đã ghi xong: 45000000 dòng (row_id đến 45000000)
Đã ghi xong: 50000000 dòng (row_id đến 50000000)
File lưu tại: ../hdfs/data-input/drug200.parquet
Tổng thời gian: 0.77 phút
